# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第5章 Praatによる音響分析 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch05_window_length.R`

対象章: 第5章「Praatによる音響分析」

「測定パラメータの設定と結果への影響」節の作例。フォルマント分析の
窓長(Praatの"window length"パラメータ)が、時間分解能と周波数分解能の
トレードオフをどう左右するかを、実際に計算して確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

window_lengths <- c(0.010, 0.025, 0.050)   # 秒

# 周波数分解能の目安: およそ 1 / 窓長(Hz)
freq_resolution <- 1 / window_lengths

# 時間分解能の目安: 窓の半分だけ前後にずれると値が変わりうる(ミリ秒換算)
time_resolution_ms <- window_lengths / 2 * 1000

d <- data.frame(
  window_length_s   = window_lengths,
  freq_resolution_hz = round(freq_resolution, 1),
  time_resolution_ms = time_resolution_ms
)
print(d)


## `ch05_pitch_range.R`

対象章: 第5章「Praatによる音響分析」

「測定パラメータの設定と結果への影響」節の作例。ピッチレンジ(floor/
ceiling)の設定が狭すぎる・広すぎる場合に、真のF0がその範囲を外れる
ことでオクターブエラーが生じる様子を、架空のF0軌跡で確認する。
実測データではない。自己相関法に基づくピッチ追跡アルゴリズムでは、
真のF0がceilingを超えると、その半分の周波数(1オクターブ下)が
誤って検出されやすいことが知られている。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_points <- 500
t <- seq(0, 2, length.out = n_points)

# 自然な発話のF0軌跡を模した架空の曲線: 全体的な下降(declination)に、
# 強調によるピーク(260Hz前後まで上昇)と、句末の低下(60Hz台まで下降)を重ねる。
true_f0 <- 170 - 30 * t +
  110 * exp(-((t - 0.6) / 0.15)^2) +     # 強調によるピーク
  -50 * exp(-((t - 1.8) / 0.1)^2) +      # 句末の低下
  rnorm(n_points, 0, 4)

settings <- tibble::tribble(
  ~setting,   ~floor, ~ceiling,
  "狭い",      100,     250,
  "標準",      75,      350,
  "広い",      50,      600
)

# 設定ごとに、真のF0が範囲外になる(オクターブエラーが生じうる)割合を計算し、
# 範囲外の点について、典型的な自己相関法の挙動(ceiling超過は半分の周波数、
# floor未満は追跡失敗)を模して「測定される」F0を生成する。
simulate_measured <- function(floor, ceiling) {
  measured <- true_f0
  above <- true_f0 > ceiling
  below <- true_f0 < floor
  measured[above] <- true_f0[above] / 2       # ceiling超過 → 半分の周波数として誤検出
  measured[below] <- NA                        # floor未満 → 追跡失敗(undefined)
  list(measured = measured, pct_error = mean(above | below) * 100)
}

results <- settings |>
  rowwise() |>
  mutate(pct_error = simulate_measured(floor, ceiling)$pct_error) |>
  ungroup()

print(results)

d_plot <- purrr::pmap_dfr(settings, function(setting, floor, ceiling) {
  m <- simulate_measured(floor, ceiling)$measured
  tibble(setting = setting, t = t, f0 = m)
}) |>
  mutate(setting = factor(setting, levels = c("狭い", "標準", "広い")))

d_true <- tibble(t = t, f0 = true_f0)

p <- ggplot(d_plot, aes(t, f0)) +
  geom_line(data = d_true, aes(t, f0), colour = "grey70", linewidth = 0.4) +
  geom_point(size = 0.6, colour = "#A8681F") +
  facet_wrap(~setting, nrow = 1) +
  labs(x = "時間 (s)", y = "F0 (Hz)",
       caption = "灰色は真のF0軌跡、点は各ピッチレンジ設定での測定値(シミュレーション)。scripts/R/ch05_pitch_range.R より生成。")
save_fig(p, "fig05-03-pitch-range-comparison", chapter = "ch05", width = 9, height = 3)

cat("完了。図はassets/figures/ch05/に保存した。\n")


## `ch05_boundary_agreement.R`

対象章: 第5章「Praatによる音響分析」

「アノテーター間の一致率」節の作例。境界時刻のような連続的な測定について、
2人のアノテーターの一致を平均絶対誤差(MAE)で報告する場合、ランダムな
ばらつきだけでなく系統的なずれ(バイアス)がある場合にMAEがどう振る舞うか
を、実際にシミュレーションで確認する。架空のデータであり、実測データ
ではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_boundaries <- 100
true_boundary <- rnorm(n_boundaries, 500, 80)   # ミリ秒、真の境界時刻(架空)

# ケース1: ランダムなばらつきのみ(系統的なずれなし)
annot_a_random <- true_boundary + rnorm(n_boundaries, 0, 8)
annot_b_random <- true_boundary + rnorm(n_boundaries, 0, 8)
diff_random <- annot_a_random - annot_b_random

# ケース2: アノテーターBが常に一定時間早く境界を引く(系統的バイアス)
annot_a_biased <- true_boundary + rnorm(n_boundaries, 0, 8)
annot_b_biased <- true_boundary - 15 + rnorm(n_boundaries, 0, 8)   # 常に15ms早い
diff_biased <- annot_a_biased - annot_b_biased

cat("--- ランダムなばらつきのみ ---\n")
cat("MAE:", round(mean(abs(diff_random)), 1), "ms  平均差(符号つき):",
    round(mean(diff_random), 1), "ms\n")

cat("\n--- 系統的バイアスあり(Bが常に15ms早い) ---\n")
cat("MAE:", round(mean(abs(diff_biased)), 1), "ms  平均差(符号つき):",
    round(mean(diff_biased), 1), "ms\n")
